# Global Climate Space

Shows how the 1000 medoids land in **climate space**, from the monthly
**CHELSA v2.1** climatologies (1981-2010), masked to the **GMBA** mountain regions.

- Sections 1-4 build the annual climate, sample the medoids and a background
  cloud, and write an interactive Plotly explorer.
- Sections 5-6 render the per-variable-pair density plots (with supersite 90%
  hulls) and ingest them as image assets for the GEE app (`gee/global_sample_app.js`).

**Run order:** 1-3, wait for the background Drive export and download it to
`../data/`, then 4-6.

In [ ]:
import ee, geemap
import numpy as np, pandas as pd
from pathlib import Path

PROJECT = "promising-era-496715-j5"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("GEE ready")

## 1 - Configuration

CHELSA scale/offset map the packed integers to physical units; `agg` sets the
annual aggregation. Supersite shapefiles and the asset/bucket names live here too.

In [ ]:
CHELSA_ROOT  = f"projects/{PROJECT}/assets/chelsa_climatologies/1981-2010"
PERIOD       = "1981-2010"
GMBA_ASSET   = f"projects/{PROJECT}/assets/GMBA_Inventory_standard_300"

# CHELSA v2.1 monthly climatologies -> annual values.
#   scale/offset : packed integer -> physical units (verify against the CHELSA v2.1 spec)
#   agg          : annual aggregation (mean, or sum for water-flux variables)
CHELSA = {
    "tas":     {"scale": 0.1,  "offset": -273.15, "agg": "mean", "unit": "°C",      "label": "Mean annual air temp"},
    "tasmax":  {"scale": 0.1,  "offset": -273.15, "agg": "mean", "unit": "°C",      "label": "Max temperature"},
    "tasmin":  {"scale": 0.1,  "offset": -273.15, "agg": "mean", "unit": "°C",      "label": "Min temperature"},
    "pr":      {"scale": 0.1,  "offset": 0.0,     "agg": "sum",  "unit": "mm/yr",   "label": "Annual precipitation"},
    "pet":     {"scale": 0.01, "offset": 0.0,     "agg": "sum",  "unit": "mm/yr",   "label": "Annual PET"},
    "cmi":     {"scale": 0.1,  "offset": 0.0,     "agg": "sum",  "unit": "kg/m²",   "label": "Climate moisture index"},
    "hurs":    {"scale": 0.01, "offset": 0.0,     "agg": "mean", "unit": "%",       "label": "Relative humidity"},
    "clt":     {"scale": 0.01, "offset": 0.0,     "agg": "mean", "unit": "%",       "label": "Cloud cover"},
    "rsds":    {"scale": 0.1,  "offset": 0.0,     "agg": "mean", "unit": "MJ/m²/d", "label": "Shortwave radiation"},
    "sfcWind": {"scale": 0.01, "offset": 0.0,     "agg": "mean", "unit": "m/s",     "label": "Wind speed"},
    "vpd":     {"scale": 0.1,  "offset": 0.0,     "agg": "mean", "unit": "Pa",      "label": "Vapour pressure deficit"},
}
VARS = list(CHELSA)

SAMPLE_SCALE = 1000          # m (CHELSA native ~1 km)
DRIVE_FOLDER = "MountAInWater"
BG_ASSET     = f"projects/{PROJECT}/assets/global_mountain_sample"   # 50k stratified sample
BG_PREFIX    = "climate_space_bg"
MEDOID_CSV   = "global_mountain_sample_1000.csv"

# App plots (sections 5-6): render -> stage in GCS -> ingest as image assets.
GCS_BUCKET   = "chelsa"          # GCS staging bucket (any region works for ingestion)
GCS_PREFIX   = "climate_space"
PLOT_FOLDER  = f"projects/{PROJECT}/assets/climate_space_plots"   # ingested RGB plots (b1,b2,b3)

# Supersite basins drawn as 90% climate hulls on the plots and as a polygon asset.
# crs=None reads the CRS from the shapefile's .prj; set an EPSG if the .prj is missing.
SUPERSITES = {
    "Pamir":     {"shp": r"C:\Users\stefan\Projects\MountAInWater\Supersites\Pamirs\Large_basin_Rogun\Basin_Fedchenko_250m.shp",       "crs": 32642, "color": "#000000"},
    "Riosanta":  {"shp": r"C:\Users\stefan\Projects\MountAInWater\Supersites\Peru\Riosanta.shp",                                       "crs": None,  "color": "#3cb043"},
    "Vilcanota": {"shp": r"C:\Users\stefan\Projects\MountAInWater\Supersites\Peru\Vilcanota.shp",                                      "crs": None,  "color": "#cc00cc"},
    "Trisuli":   {"shp": r"C:\Users\stefan\Projects\MountAInWater\Supersites\Himalaya\Trisuli_Supersite\trisuli_supersite_UTM45N.shp", "crs": None,  "color": "#8c510a"},
}
SUP_ASSET = f"projects/{PROJECT}/assets/supersites"

OUT_DIR = Path("../data"); OUT_DIR.mkdir(exist_ok=True)

## 2 - Annual CHELSA climate image (masked to GMBA)

Each variable is the annual aggregate of its 12 monthly assets, scaled to physical units, then masked to the GMBA mountain regions.

In [ ]:
def chelsa_annual(var):
    cfg    = CHELSA[var]
    months = [ee.Image(f"{CHELSA_ROOT}/{var}/CHELSA_{var}_{m:02d}_{PERIOD}_V21")
                .select(0).rename(var)          # unify band name across months
                .multiply(cfg["scale"]).add(cfg["offset"])
              for m in range(1, 13)]
    ic     = ee.ImageCollection(months)
    annual = ic.sum() if cfg["agg"] == "sum" else ic.mean()
    return annual.rename(var)

climate  = ee.Image.cat([chelsa_annual(v) for v in VARS])

gmba     = ee.FeatureCollection(GMBA_ASSET)
gmbaMask = ee.Image().byte().paint(gmba, 1)
climate  = climate.updateMask(gmbaMask)

print("Annual climate bands:", climate.bandNames().getInfo())

## 3a - Climate at the 1000 medoids

Built from the local medoid CSV, sampled on the climate image, pulled with `getInfo` (1000 < the 5000-element limit). Kept in `med_df`, also saved to disk.

In [ ]:
med_src = pd.read_csv(OUT_DIR / MEDOID_CSV)
feats   = [ee.Feature(ee.Geometry.Point([float(r.longitude), float(r.latitude)]),
                      {"kapos_class": int(r.kapos_class)})
           for r in med_src.itertuples()]
med_fc  = ee.FeatureCollection(feats)

samp   = climate.sampleRegions(collection=med_fc, scale=SAMPLE_SCALE, tileScale=8)
med_df = pd.DataFrame([f["properties"] for f in samp.getInfo()["features"]])
med_df.to_csv(OUT_DIR / "climate_space_medoids.csv", index=False)
print(f"{len(med_df)} medoids with climate")
med_df.head()

## 3b - Background climate density -> Drive

Samples the climate image at the 50k `global_mountain_sample` points and exports
to Drive (too large for `getInfo`). Download `climate_space_bg.csv` into `../data/`
before sections 4-5. The background reflects the stratified sample, i.e. the
sampled climate envelope rather than the area-true density.

In [ ]:
bg_pts = ee.FeatureCollection(BG_ASSET)
bg     = climate.sampleRegions(collection=bg_pts, scale=SAMPLE_SCALE, tileScale=8)

task = ee.batch.Export.table.toDrive(
    collection = bg, description = BG_PREFIX, folder = DRIVE_FOLDER,
    fileNamePrefix = BG_PREFIX, fileFormat = "CSV", selectors = VARS)
task.start()
print("Background export:", task.id, f"-> Drive/{DRIVE_FOLDER}/{BG_PREFIX}.csv")
print("Monitor: https://code.earthengine.google.com/tasks")

## 4 - Interactive climate-space explorer

Precomputes a 2D-density histogram for every variable pair and overlays the
medoids (Kapos colour), with an X|Y dropdown. Writes a self-contained HTML.

Needs `climate_space_bg.csv` (section 3b) downloaded into `../data/`.

In [ ]:
import plotly.graph_objects as go
from itertools import combinations

bg_df = pd.read_csv(OUT_DIR / f"{BG_PREFIX}.csv").dropna(subset=VARS)
try:
    mdf = med_df.dropna(subset=VARS)
except NameError:
    mdf = pd.read_csv(OUT_DIR / "climate_space_medoids.csv").dropna(subset=VARS)

VAR_AX     = {v: f"{CHELSA[v]['label']} ({CHELSA[v]['unit']})" for v in VARS}
pairs      = list(combinations(VARS, 2))
KAPOS_HEX  = ["#54278f", "#2171b5", "#08519c", "#1a9641", "#fee08b", "#f46d43"]
kapos_vals = sorted(int(c) for c in mdf["kapos_class"].unique())
BINS       = 40
tpp        = 1 + len(kapos_vals)            # traces per pair: heatmap + one scatter/class

traces = []
for xv, yv in pairs:
    counts, xe, ye = np.histogram2d(bg_df[xv], bg_df[yv], bins=BINS)
    traces.append(go.Heatmap(
        x=0.5 * (xe[:-1] + xe[1:]), y=0.5 * (ye[:-1] + ye[1:]),
        z=np.log1p(counts.T), colorscale="YlOrRd", showscale=False, visible=False,
        hovertemplate=f"{xv}: %{{x:.3g}}<br>{yv}: %{{y:.3g}}<extra></extra>"))
    for cls in kapos_vals:
        m = mdf["kapos_class"] == cls
        traces.append(go.Scatter(
            x=mdf.loc[m, xv], y=mdf.loc[m, yv], mode="markers",
            marker=dict(color=KAPOS_HEX[cls - 1], size=6, line=dict(color="black", width=0.4)),
            name=f"K{cls}", visible=False,
            showlegend=(xv == pairs[0][0] and yv == pairs[0][1])))

for k in range(tpp):
    traces[k].visible = True


def vis(idx):
    v = [False] * len(traces)
    for k in range(tpp):
        v[idx * tpp + k] = True
    return v


buttons = [dict(label=f"{xv}  |  {yv}", method="update",
                args=[{"visible": vis(i)},
                      {"xaxis.title.text": VAR_AX[xv], "yaxis.title.text": VAR_AX[yv]}])
           for i, (xv, yv) in enumerate(pairs)]

fig = go.Figure(data=traces)
fig.update_layout(
    title="Mountain climate space - 1000 medoids over background density",
    xaxis_title=VAR_AX[pairs[0][0]], yaxis_title=VAR_AX[pairs[0][1]],
    updatemenus=[dict(buttons=buttons, direction="down", x=0.0, xanchor="left",
                      y=1.15, yanchor="top", bgcolor="white", bordercolor="#bbb")],
    legend=dict(title="Kapos", x=1.02, y=1, xanchor="left"),
    width=780, height=600, plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=70, r=120, t=90, b=60))
fig.update_xaxes(showgrid=True, gridcolor="#f0f0f0", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0", zeroline=False)

html = OUT_DIR / "climate_space_explorer.html"
fig.write_html(str(html), include_plotlyjs="cdn")
print(f"Saved {html}  ({len(pairs)} variable pairs)")
fig.show()

## 5 - Render the app plots and stage them in GCS

Renders both orientations of every variable pair (110) as a hexbin density of the
background, the 1000 medoids by Kapos class, and a **90% climate hull** for each
supersite basin. Each plot is written as a GeoTIFF to
`gs://{GCS_BUCKET}/{GCS_PREFIX}/` (staging for section 6), and the supersite
polygons are uploaded as the `supersites` FeatureCollection asset for the map.

Needs `climate_space_bg.csv` (section 3b, downloaded) and `climate_space_medoids.csv`
(section 3a) in `../data/`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import geopandas as gpd
from itertools import permutations
from scipy.stats import gaussian_kde
from scipy.spatial import ConvexHull
import rasterio
from rasterio.transform import from_origin
from google.cloud import storage

TIF_DIR   = OUT_DIR / "climate_space_tif"; TIF_DIR.mkdir(exist_ok=True)
KAPOS_HEX = ["#54278f", "#2171b5", "#08519c", "#1a9641", "#fee08b", "#f46d43"]

bg_df = pd.read_csv(OUT_DIR / f"{BG_PREFIX}.csv").dropna(subset=VARS)
mdf   = pd.read_csv(OUT_DIR / "climate_space_medoids.csv").dropna(subset=VARS)
kapos_vals = sorted(int(c) for c in mdf["kapos_class"].unique())

# Sample CHELSA inside each supersite basin; upload the polygons as one asset.
site_df, feats = {}, []
for name, cfg in SUPERSITES.items():
    g = gpd.read_file(cfg["shp"])
    if cfg["crs"]:
        g = g.set_crs(cfg["crs"], allow_override=True)
    geom = ee.Geometry(g.to_crs(4326).geometry.union_all().simplify(0.003).__geo_interface__)
    s    = climate.sample(region=geom, scale=SAMPLE_SCALE, numPixels=3000, dropNulls=True)
    site_df[name] = pd.DataFrame([f["properties"] for f in s.getInfo()["features"]]).dropna(subset=VARS)
    feats.append(ee.Feature(geom, {"site": name, "color": cfg["color"]}))
    print(f"{name}: {len(site_df[name])} climate pixels")

try:
    ee.data.deleteAsset(SUP_ASSET)
except Exception:
    pass
ee.batch.Export.table.toAsset(
    collection=ee.FeatureCollection(feats), description="supersites", assetId=SUP_ASSET).start()
print("supersites asset export submitted ->", SUP_ASSET)


def hull90(ax, x, y, color, label):
    """Convex hull of the densest 90% of a basin's climate pixels."""
    pts = np.column_stack([x, y])
    if len(pts) >= 20 and np.ptp(x) > 0 and np.ptp(y) > 0:
        d = gaussian_kde(np.vstack([x, y]))(np.vstack([x, y]))
        pts = pts[d >= np.quantile(d, 0.10)]
    if len(pts) >= 3:
        ring = np.vstack([pts[ConvexHull(pts).vertices], pts[ConvexHull(pts).vertices][0]])
        ax.plot(ring[:, 0], ring[:, 1], color=color, lw=2.2, label=label)
    else:
        ax.scatter(pts[:, 0], pts[:, 1], s=8, c=color, label=label)


def render_pair(xv, yv, png_path):
    fig, ax = plt.subplots(figsize=(6.0, 4.8), dpi=200)
    ax.hexbin(bg_df[xv], bg_df[yv], gridsize=50, cmap="Blues", bins="log", mincnt=1, linewidths=0)
    for cls in kapos_vals:
        m = mdf["kapos_class"] == cls
        ax.scatter(mdf.loc[m, xv], mdf.loc[m, yv], s=16, c=KAPOS_HEX[cls - 1],
                   edgecolors="black", linewidths=0.3, label=f"K{cls}")
    for name, cfg in SUPERSITES.items():
        hull90(ax, site_df[name][xv].values, site_df[name][yv].values, cfg["color"], name)
    ax.set_xlabel(f"{CHELSA[xv]['label']} ({CHELSA[xv]['unit']})", fontsize=14)
    ax.set_ylabel(f"{CHELSA[yv]['label']} ({CHELSA[yv]['unit']})", fontsize=14)
    ax.tick_params(labelsize=11)
    ax.legend(fontsize=13, framealpha=0.7, loc="best", ncol=3, markerscale=1.3)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.close(fig)


def png_to_cog(png_path, tif_path):
    with rasterio.open(png_path) as src:
        arr = src.read()[:3]                       # RGB, drop alpha
    _, h, w = arr.shape
    transform = from_origin(0, h * 100, 100, 100)  # dummy grid (display only)
    with rasterio.open(tif_path, "w", driver="COG", height=h, width=w, count=3,
                       dtype="uint8", crs="EPSG:3857", transform=transform, compress="DEFLATE") as dst:
        dst.write(arr)


client = storage.Client(project=PROJECT, credentials=ee.data.get_persistent_credentials())
try:
    bucket = client.get_bucket(GCS_BUCKET)
except Exception:
    bucket = client.create_bucket(GCS_BUCKET)

pairs = list(permutations(VARS, 2))             # both orientations -> 110
for xv, yv in pairs:
    key = f"{xv}__{yv}"
    png, tif = TIF_DIR / f"{key}.png", TIF_DIR / f"{key}.tif"
    render_pair(xv, yv, png)
    png_to_cog(png, tif)
    bucket.blob(f"{GCS_PREFIX}/{key}.tif").upload_from_filename(str(tif))

print(f"Rendered + staged {len(pairs)} GeoTIFFs -> gs://{GCS_BUCKET}/{GCS_PREFIX}/")

## 6 - Ingest the plots as image assets

Ingests each staged GeoTIFF into `climate_space_plots/{x}__{y}` (RGB bands
b1,b2,b3) — what the app shows with `ui.Thumbnail`. ~110 tasks; `allow_overwrite`
lets you re-run without deleting first.

In [ ]:
from itertools import permutations

# Create the destination folder asset (ignore if it already exists).
try:
    ee.data.createAsset({"type": "FOLDER"}, PLOT_FOLDER)
    print("Created folder", PLOT_FOLDER)
except Exception:
    pass

pairs = [f"{a}__{b}" for a, b in permutations(VARS, 2)]
for key in pairs:
    rid = ee.data.newTaskId()[0]
    ee.data.startIngestion(rid, {
        "id":       f"{PLOT_FOLDER}/{key}",
        "tilesets": [{"sources": [{"uris": [f"gs://{GCS_BUCKET}/{GCS_PREFIX}/{key}.tif"]}]}],
        "properties": {"pair": key},
    }, allow_overwrite=True)

print(f"Submitted {len(pairs)} ingestion tasks -> {PLOT_FOLDER}/{{x}}__{{y}} (bands b1,b2,b3)")
print("Monitor: https://code.earthengine.google.com/tasks")